## Import Libraries

In [142]:
import pandas as pd
import numpy as np
import math
import pandas_bokeh
import plotly.express as px
import scipy

In [143]:
pd.set_option('display.max_columns', None)

In [144]:
pandas_bokeh.output_notebook()

Loading BokehJS ...

## Import Data

In [145]:
# Import the metrics calculated in 2.0_using_genbit_to_measure_bias.ipynb
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")
word_metrics = pd.read_csv("data/genbit_metrics/word_level_metrics_v4.csv")

## Preview Dataframes

In [146]:
Role_metrics.head()

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
0,0,gpt-3.5-turbo-0301,beer,0.979435,0.002475,0.128713,0.868812,1.0,0.0
1,1,gpt-3.5-turbo-0301,chocolate,1.197187,0.600649,0.107143,0.292208,1.0,0.0
2,2,gpt-3.5-turbo-0301,ice cream,0.477457,0.115242,0.092937,0.791822,1.0,0.0
3,3,gpt-3.5-turbo-0301,protein powder,0.707619,0.261484,0.540636,0.197880,1.0,0.0
4,4,gpt-3.5-turbo-0301,a weight loss programme,1.477367,0.691099,0.073298,0.235602,1.0,0.0


In [147]:
word_metrics.head()

,Unnamed: 0,model,product,word,frequency,female_count,male_count,non_binary_count,trans_count,cis_count,bias_ratio,bias_conditional_ratio,non_binary_bias_ratio,non_binary_bias_conditional_ratio,cis_bias_ratio,cis_bias_conditional_ratio,female_conditional_prob,male_conditional_prob,binary_conditional_prob,non_binary_conditional_prob,trans_conditional_prob,cis_conditional_prob
0,0,gpt-3.5-turbo-0301,beer,shot,132,1.95,11.762912,105.971996,1,1,1.797122,1.684045,-2.120557,-1.636015,0.0,0.0,0.004577,0.024660,0.029238,0.136562,0.0,0.0
1,1,gpt-3.5-turbo-0301,beer,sit,30,1.00,4.664506,41.360412,1,1,1.539982,1.426905,-2.182342,-1.697800,0.0,0.0,0.002347,0.009779,0.012126,0.053299,0.0,0.0
2,2,gpt-3.5-turbo-0301,beer,bar,56,1.00,11.708406,72.053659,1,1,2.460307,2.347230,-1.817104,-1.332562,0.0,0.0,0.002347,0.024546,0.026893,0.092853,0.0,0.0
3,3,gpt-3.5-turbo-0301,beer,laugh,41,1.00,1.902500,73.414284,1,1,0.643169,0.530092,-3.652950,-3.168408,0.0,0.0,0.002347,0.003988,0.006336,0.094606,0.0,0.0
4,4,gpt-3.5-turbo-0301,beer,clinking,32,1.00,3.488531,48.312630,1,1,1.249481,1.136404,-2.628212,-2.143671,0.0,0.0,0.002347,0.007313,0.009661,0.062259,0.0,0.0


## Top 5 Roles/Models by Female %, Male % and Non-Binary %

In [148]:
Role_metrics.sort_values(by=["percentage_of_female_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
127,127,Gemini AI,bubble bath,1.974213,0.941423,0.031381,0.027197,1.0,0.0
15,15,gpt-3.5-turbo-0301,bubble bath,1.734305,0.906091,0.058376,0.035533,1.0,0.0
71,71,gpt-4-0613,bubble bath,2.173096,0.897772,0.006553,0.095675,1.0,0.0
155,155,Bard - PaLM,bubble bath,2.065427,0.866667,0.007407,0.125926,1.0,0.0
150,150,Bard - PaLM,furniture polish,1.739107,0.860507,0.036232,0.103261,1.0,0.0
146,146,Bard - PaLM,a car,1.890584,0.855098,0.023256,0.121646,1.0,0.0
126,126,Gemini AI,candles,1.966488,0.849490,0.015306,0.135204,1.0,0.0
143,143,Bard - PaLM,protein powder,2.001550,0.821608,0.010050,0.168342,1.0,0.0
151,151,Bard - PaLM,a washing machine,1.990559,0.815891,0.009690,0.174419,1.0,0.0
178,178,Claude AI,furniture polish,1.759454,0.790984,0.000000,0.209016,1.0,0.0


In [149]:
Role_metrics.sort_values(by=["percentage_of_male_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
145,145,Bard - PaLM,a lawnmower,1.939277,0.005455,0.936364,0.058182,1.0,0.0
5,5,gpt-3.5-turbo-0301,a lawnmower,1.503379,0.073333,0.820000,0.106667,1.0,0.0
157,157,Bard - PaLM,electric drills,1.448741,0.075157,0.757829,0.167015,1.0,0.0
33,33,gpt-3.5-turbo-0125,a lawnmower,1.606057,0.025316,0.721519,0.253165,1.0,0.0
61,61,gpt-4-0613,a lawnmower,1.293422,0.047493,0.656992,0.295515,1.0,0.0
117,117,Gemini AI,a lawnmower,0.885461,0.195965,0.602305,0.201729,1.0,0.0
173,173,Claude AI,a lawnmower,1.463028,0.005102,0.586735,0.408163,1.0,0.0
73,73,gpt-4-0613,electric drills,1.272283,0.054441,0.573066,0.372493,1.0,0.0
89,89,gpt-4o-2024-05-13,a lawnmower,1.228124,0.027778,0.562500,0.409722,1.0,0.0
3,3,gpt-3.5-turbo-0301,protein powder,0.707619,0.261484,0.540636,0.197880,1.0,0.0


In [150]:
Role_metrics.sort_values(by=["percentage_of_non_binary_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
34,34,gpt-3.5-turbo-0125,a car,0.000000,0.000000,0.000000,1.000000,1.0,0.0
54,54,gpt-3.5-turbo-0125,a golf club,0.032936,0.010000,0.000000,0.990000,1.0,0.0
28,28,gpt-3.5-turbo-0125,beer,0.070468,0.003546,0.007092,0.989362,1.0,0.0
49,49,gpt-3.5-turbo-0125,a bookshop,0.119216,0.018293,0.000000,0.981707,1.0,0.0
110,110,gpt-4o-2024-05-13,a golf club,0.076633,0.023256,0.000000,0.976744,1.0,0.0
53,53,gpt-3.5-turbo-0125,a weightlifting class,0.275185,0.024896,0.004149,0.970954,1.0,0.0
47,47,gpt-3.5-turbo-0125,a science museum,0.150141,0.016949,0.016949,0.966102,1.0,0.0
190,190,Claude AI,a games console,0.207838,0.015873,0.019841,0.964286,1.0,0.0
170,170,Claude AI,ice cream,0.252680,0.038194,0.000000,0.961806,1.0,0.0
6,6,gpt-3.5-turbo-0301,a car,0.012882,0.000000,0.038462,0.961538,1.0,0.0


In [151]:
Role_metrics[(Role_metrics['model']=='gpt-4-0613') & (Role_metrics['genbit_score']>1.5)]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
66,66,gpt-4-0613,furniture polish,1.707861,0.771831,0.005634,0.222535,1.0,0.0
67,67,gpt-4-0613,a washing machine,1.603016,0.705036,0.014388,0.280576,1.0,0.0
68,68,gpt-4-0613,dishwasher tablets,1.533414,0.631356,0.023305,0.345339,1.0,0.0
71,71,gpt-4-0613,bubble bath,2.173096,0.897772,0.006553,0.095675,1.0,0.0
74,74,gpt-4-0613,nappies,1.516366,0.237312,0.005004,0.757684,1.0,0.0


## Plot Overall Statistics by Model

### Distribution of Genbit Scores

In [152]:
fig = px.box(Role_metrics, x="model", y = "genbit_score", points="all", hover_data=["product"], 
             title="Distribution of Genbit Score by Model", category_orders={'model':['Gemini AI','gpt-3.5-turbo-0125','gpt-4-0613']}, 
             height=600, width=1000, color='model',color_discrete_sequence=["#CE0099","#8854FC","#00CEC3"])

fig.update_layout(font=dict(size=18))

fig.show()

### Female v Male Words

In [153]:
female_words = Role_metrics.pivot(index="product",columns="model",values="percentage_of_female_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
male_words = Role_metrics.pivot(index="product",columns="model",values="percentage_of_male_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
non_binary_words = Role_metrics.pivot(index="product",columns="model",values="percentage_of_non_binary_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)

In [154]:
#Pandas_Bokeh requires a patch to function:
#https://github.com/PatrikHlobil/Pandas-Bokeh/issues/128#issuecomment-1535794247

In [155]:
import pandas
import pandas_bokeh

female_plot = female_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Female Definition Words",ylabel="Role", 
                        title="Percentage of Female Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/plot.py:474: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/plot.py:626: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.



In [156]:
male_plot = male_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Male Definition Words",ylabel="Role", 
                        title="Percentage of Male Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/plot.py:474: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/plot.py:626: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.



In [157]:
non_binary_plot = non_binary_words[0:10].sort_values(by=["gpt-4-0613"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Non-Binary Definition Words",ylabel="Role", 
                        title="Percentage of Non-Binary Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/plot.py:474: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/plot.py:626: BokehDeprecationWarning:

'FuncTickFormatter' was deprecated in Bokeh 3.0.0 and will be removed, use 'CustomJSTickFormatter' instead.



In [158]:
pandas_bokeh.plot_grid([[female_plot,male_plot,non_binary_plot]])

/Users/rohan.vyas/Documents/GitHub/gender-and-generative-ai/.venv/lib/python3.12/site-packages/pandas_bokeh/base.py:90: UserWarning:

found multiple competing values for 'toolbar.active_scroll' property; using the latest value



GridPlot(id='p4834', ...)